In [1]:
# ==============================================================
# Enhanced GEC + GLEU System (Malayalam-safe version)
# Handles missing or blank records automatically
# ==============================================================

import pandas as pd
import numpy as np
import re
from collections import defaultdict, Counter
from tqdm import tqdm

# ==============================================================
# 1. LOAD DATA
# ==============================================================

print("📘 Loading training and dev data...")
train_df = pd.read_csv('/kaggle/input/malayalam-text1/train.csv') # change to different language accordingly
dev_df = pd.read_csv('/kaggle/input/malayalam-text1/dev.csv') 

# Drop completely empty or NaN rows in both columns
train_df = train_df.dropna(subset=['Input sentence', 'Output sentence'])
dev_df = dev_df.dropna(subset=['Input Sentence', 'Output Sentence'])

# Drop rows where input or output is empty string after stripping
train_df = train_df[(train_df['Input sentence'].astype(str).str.strip() != "") &
                    (train_df['Output sentence'].astype(str).str.strip() != "")]
dev_df = dev_df[(dev_df['Input Sentence'].astype(str).str.strip() != "") &
                (dev_df['Output Sentence'].astype(str).str.strip() != "")]

train_inputs = train_df['Input sentence'].astype(str).tolist()
train_outputs = train_df['Output sentence'].astype(str).tolist()
dev_inputs = dev_df['Input Sentence'].astype(str).tolist()
dev_outputs = dev_df['Output Sentence'].astype(str).tolist()

print(f"✓ Loaded {len(train_inputs)} clean training samples")
print(f"✓ Loaded {len(dev_inputs)} clean dev samples")
print(f"\nExample Training Pair:\nInput: {train_inputs[0]}\nOutput: {train_outputs[0]}")

# ==============================================================
# 2. TOKENIZATION + GLEU METRIC (Smoothed)
# ==============================================================

def tokenize(text):
    """Improved multilingual tokenization (handles Indic languages well)."""
    text = re.sub(r"[“”\"'.,!?;:()\-–—/]", "", text)
    text = re.sub(r"\s+", " ", text.strip())
    return text.lower().split()

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def calculate_gleu(reference, hypothesis, max_n=4, smooth=1e-8):
    """Smoothed and weighted GLEU metric."""
    ref_tokens = tokenize(reference)
    hyp_tokens = tokenize(hypothesis)
    if not hyp_tokens:
        return 0.0

    weights = [0.4, 0.3, 0.2, 0.1]
    precisions = []

    for n in range(1, max_n + 1):
        ref_ngrams = Counter(get_ngrams(ref_tokens, n))
        hyp_ngrams = Counter(get_ngrams(hyp_tokens, n))
        matches = sum(min(count, ref_ngrams[ng]) for ng, count in hyp_ngrams.items())
        total = max(sum(hyp_ngrams.values()), 1)
        precision = (matches + smooth) / (total + smooth)
        precisions.append(precision)

    weighted_precision = sum(w * p for w, p in zip(weights, precisions))
    ref_len, hyp_len = len(ref_tokens), len(hyp_tokens)
    bp = np.exp(1 - ref_len / hyp_len) if hyp_len < ref_len else 1.0
    return bp * weighted_precision

# ==============================================================
# 3. IMPROVED GEC MODEL
# ==============================================================

class GECModel:
    def __init__(self):
        self.exact_matches = {}
        self.word_map = defaultdict(Counter)
        self.bigram_map = defaultdict(Counter)

    def train(self, inputs, outputs):
        print("\n🧠 Training model (learning word/bigram corrections)...")
        for inp, out in tqdm(zip(inputs, outputs), total=len(inputs)):
            inp_clean = inp.strip().lower()
            out_clean = out.strip()
            self.exact_matches[inp_clean] = out_clean

            inp_tokens = tokenize(inp)
            out_tokens = tokenize(out)

            if len(inp_tokens) == len(out_tokens):
                for i in range(len(inp_tokens)):
                    if inp_tokens[i] != out_tokens[i]:
                        self.word_map[inp_tokens[i]][out_tokens[i]] += 1
                        if i > 0:
                            bigram = (inp_tokens[i-1], inp_tokens[i])
                            self.bigram_map[bigram][out_tokens[i]] += 1

        print(f"✓ Learned {len(self.word_map)} word corrections")
        print(f"✓ Learned {len(self.bigram_map)} bigram corrections")

    def apply_rules(self, text):
        """Extended grammar rules (mostly neutral for Malayalam)."""
        rules = [
            (r"\ba ([aeiou])", r"an \1"),
        ]
        result = text
        for pattern, repl in rules:
            result = re.sub(pattern, repl, result, flags=re.IGNORECASE)
        return result

    def correct(self, sentence):
        sent_lower = sentence.strip().lower()
        if sent_lower in self.exact_matches:
            return self.exact_matches[sent_lower]

        corrected = self.apply_rules(sentence)
        tokens = tokenize(corrected)
        corrected_tokens = []

        freq_threshold = 2 if len(self.word_map) < 10000 else 3

        for i, token in enumerate(tokens):
            if i > 0:
                bigram = (tokens[i-1], token)
                if bigram in self.bigram_map:
                    best = self.bigram_map[bigram].most_common(1)[0]
                    if best[1] >= freq_threshold:
                        corrected_tokens.append(best[0])
                        continue

            if token in self.word_map:
                best = self.word_map[token].most_common(1)[0]
                if best[1] >= freq_threshold:
                    corrected_tokens.append(best[0])
                    continue

            corrected_tokens.append(token)

        return " ".join(corrected_tokens)

# ==============================================================
# 4. TRAIN + EVALUATE
# ==============================================================

model = GECModel()
model.train(train_inputs, train_outputs)

print("\n⚙️ Evaluating on dev set...")
gleu_scores = []
predictions = []

for inp, out in tqdm(zip(dev_inputs, dev_outputs), total=len(dev_inputs)):
    pred = model.correct(inp)
    predictions.append(pred)
    gleu = calculate_gleu(out, pred)
    gleu_scores.append(gleu)

avg_gleu = np.mean(gleu_scores)
print(f"\n🌟 FINAL AVERAGE GLEU SCORE: {avg_gleu:.4f}")

# ==============================================================
# 5. SAVE RESULTS
# ==============================================================

results_df = pd.DataFrame({
    "Input": dev_inputs,
    "Reference": dev_outputs,
    "Prediction": predictions,
    "GLEU": [f"{score:.4f}" for score in gleu_scores]
})
results_df.to_csv("improved_gec_results_malayalam.csv", index=False)

print(f"\n📁 Results saved to 'improved_gec_results_malayalam.csv'")
print(f"Average GLEU ≈ {avg_gleu:.4f}")
print("="*70)


📘 Loading training and dev data...
✓ Loaded 296 clean training samples
✓ Loaded 50 clean dev samples

Example Training Pair:
Input: ഇന്നത്തെ കാലഘട്ടത്തിൽ നമുക്ക് ഒഴിച്ചുകൂടാൻ പറ്റാത്ത ഒന്നാണ് സാമൂഹ്യമാധ്യമങ്ങൾ.
Output: ഇന്നത്തെ കാലഘട്ടത്തിൽ നമുക്ക് ഒഴിച്ചുകൂടാൻ പറ്റാത്ത ഒന്നാണ് സമൂഹമാധ്യമങ്ങൾ.

🧠 Training model (learning word/bigram corrections)...


100%|██████████| 296/296 [00:00<00:00, 38147.61it/s]


✓ Learned 213 word corrections
✓ Learned 207 bigram corrections

⚙️ Evaluating on dev set...


100%|██████████| 50/50 [00:00<00:00, 8166.16it/s]


🌟 FINAL AVERAGE GLEU SCORE: 0.7450

📁 Results saved to 'improved_gec_results_malayalam.csv'
Average GLEU ≈ 0.7450
